Exclusive PTX samples of vitek, phoenix and microScan

In [ ]:
import pandas as pd
import os

df_vitek = pd.read_csv(
    "/groups/ds/Win-KID/BVBRC/BVBRC_devices/bin/VITEK_combined_02_26_interpreted_bin.csv"
)
df_phoenix = pd.read_csv(
    "/groups/ds/Win-KID/BVBRC/BVBRC_devices/bin/phoenix_combined_02_26_interpreted_bin.csv"
)
df_microScan = pd.read_csv(
    "/groups/ds/Win-KID/BVBRC/BVBRC_devices/bin/MicroScan_combined_02_26_interpreted_bin.csv"
)

available_ids_vitek_ = {
    os.path.splitext(f)[0]
    for f in os.listdir("/groups/ds/Win-KID/BVBRC/VITEK/gff_card_401")
    if f.endswith(".gff")
}
available_ids_phoenix = {
    os.path.splitext(f)[0]
    for f in os.listdir("/groups/ds/Win-KID/BVBRC/phoenix/gff_card_401")
    if f.endswith(".gff")
}
available_ids_microScan = {
    os.path.splitext(f)[0]
    for f in os.listdir("/groups/ds/Win-KID/BVBRC/MicroScan/gff_card_401")
    if f.endswith(".gff")
}

# Remove non PTX samples
df_vitek = df_vitek[df_vitek["Organism_Code"] == "PTX"]
df_phoenix = df_phoenix[df_phoenix["Organism_Code"] == "PTX"]
df_microScan = df_microScan[df_microScan["Organism_Code"] == "PTX"]

# Remove unavialible IDs
df_vitek = df_vitek[
    df_vitek["Sample_ID_IfH"].astype(str).isin(available_ids_vitek_)
].reset_index(drop=True)
df_phoenix = df_phoenix[
    df_phoenix["Sample_ID_IfH"].astype(str).isin(available_ids_phoenix)
].reset_index(drop=True)
df_microScan = df_microScan[
    df_microScan["Sample_ID_IfH"].astype(str).isin(available_ids_microScan)
].reset_index(drop=True)

df_vitek_filtered = pd.read_csv(
    "/groups/ds/Win-KID/BVBRC/BVBRC_devices/PTX/vitek_filtered.csv"
)
df_phoenix_filtered = pd.read_csv(
    "/groups/ds/Win-KID/BVBRC/BVBRC_devices/PTX/phoenix_filtered.csv"
)
df_microScan_filtered = pd.read_csv(
    "/groups/ds/Win-KID/BVBRC/BVBRC_devices/PTX/microScan_filtered.csv"
)

df_vitek_extra = df_vitek[
    ~df_vitek["Sample_ID_IfH"].isin(df_vitek_filtered["Sample_ID_IfH"])
]
df_phoenix_extra = df_phoenix[
    ~df_phoenix["Sample_ID_IfH"].isin(df_phoenix_filtered["Sample_ID_IfH"])
]
df_microScan_extra = df_microScan[
    ~df_microScan["Sample_ID_IfH"].isin(df_microScan_filtered["Sample_ID_IfH"])
]

df_vitek_extra = df_vitek_extra.dropna(subset=df_vitek_extra.columns[2:], how="all")
df_phoenix_extra = df_phoenix_extra.dropna(
    subset=df_phoenix_extra.columns[2:], how="all"
)
df_microScan_extra = df_microScan_extra.dropna(
    subset=df_microScan_extra.columns[2:], how="all"
)

df_vitek_extra.to_csv(
    "/groups/ds/Win-KID/BVBRC/BVBRC_devices/PTX/vitek_extra.csv", index=False
)
df_phoenix_extra.to_csv(
    "/groups/ds/Win-KID/BVBRC/BVBRC_devices/PTX/phoenix_extra.csv", index=False
)
df_microScan_extra.to_csv(
    "/groups/ds/Win-KID/BVBRC/BVBRC_devices/PTX/microScan_extra.csv", index=False
)

In [4]:
import pandas as pd

df_vitek_filtered = pd.read_csv(
    "/groups/ds/Win-KID/BVBRC/BVBRC_devices/PTX/vitek_filtered.csv"
)
df_phoenix_filtered = pd.read_csv(
    "/groups/ds/Win-KID/BVBRC/BVBRC_devices/PTX/phoenix_filtered.csv"
)
df_microScan_filtered = pd.read_csv(
    "/groups/ds/Win-KID/BVBRC/BVBRC_devices/PTX/microScan_filtered.csv"
)


def count_s_r_per_antibiotic(df: pd.DataFrame) -> pd.DataFrame:
    # Identify antibiotic columns
    antibiotic_cols = [
        col for col in df.columns if col not in ["Sample_ID_IfH", "Organism_Code"]
    ]

    # Count S and R for each antibiotic
    sr_counts = pd.DataFrame(
        {
            "S": (df[antibiotic_cols] == "S").sum(),
            "R": (df[antibiotic_cols] == "R").sum(),
        }
    )

    # Calculate total number of available results
    sr_counts["Total"] = sr_counts["S"] + sr_counts["R"]

    # Remove antibiotics without any S or R result
    sr_counts = sr_counts[sr_counts["Total"] > 0]

    # Convert antibiotic names from index to column
    sr_counts = sr_counts.reset_index()
    sr_counts = sr_counts.rename(columns={"index": "Antibiotic"})

    return sr_counts


vitek_count = count_s_r_per_antibiotic(df_vitek_filtered)
print("VITEK")
print(vitek_count)
vitek_count.to_csv("vitek_count.csv", index=False)

phoenix_count = count_s_r_per_antibiotic(df_phoenix_filtered)
print("phoenix")
print(phoenix_count)
phoenix_count.to_csv("phoenix_count.csv", index=False)

microScan_count = count_s_r_per_antibiotic(df_microScan_filtered)
print("microScan")
print(microScan_count)
microScan_count.to_csv("microScan_count.csv", index=False)

VITEK
      Antibiotic    S    R  Total
0       amikacin    0    4      4
1  ciprofloxacin   21  157    178
2     gentamicin   30  187    217
3   levofloxacin   16  160    176
4     tobramycin  140   77    217
phoenix
      Antibiotic   S    R  Total
0       amikacin   0    4      4
1  ciprofloxacin  20  158    178
2     gentamicin  25  192    217
3   levofloxacin  17  159    176
4     tobramycin  61  156    217
microScan
      Antibiotic   S    R  Total
0       amikacin   0    4      4
1  ciprofloxacin  18  160    178
2     gentamicin  19  198    217
3   levofloxacin   0  176    176
4     tobramycin  52  165    217
